In [1]:
DATA_SOURCES = ["archeology"]

# Setup

## Processor-Specific

In [2]:
%load_ext autoreload
%autoreload 2
import os
import sys
from dotenv import load_dotenv
from torch.backends import cudnn

# enforce more deterministic behavior
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

sys.path.append("..")
load_dotenv("../../.env")

from processor.core.interaction_conductor.chat_interface import ChatInterface, ChatInterfaceOutputFormat
from processor.model.interface.impl.gpt import GPT
from processor.core.ir_system.ir_data_model import convert_multi_retriever_results_to_str

/home/luthfi/miniconda3/envs/pneuma/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
llm_path = "model/weight/qwen3-8b"
embed_model_path = "model/weight/bge-base"
chat_interface = ChatInterface(llm_path, embed_model_path, "llm", DATA_SOURCES)
gpt = GPT("gpt-4o-mini")

[2025-08-02 17:00:09] INFO in llm_planner: Initializing LLMPlanner, the core component of Materializer Engine


## Benchmark-Specific

In [4]:
import json
def read_jsonl(file_path):
    """
    Reads a JSON Lines (.jsonl) file and returns a list of Python dictionaries.
    
    Args:
        file_path (str): Path to the JSONL file.
    
    Returns:
        list: A list of dictionaries, one per line in the file.
    """
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data
benchmark = read_jsonl(f"../../benchmark/benchmark_{DATA_SOURCES[0]}.jsonl")
def write_jsonl(filepath, data, append=False):
    """
    Write a list of JSON-serializable objects to a JSONL file.

    Args:
        filepath (str): Path to the output file.
        data (list): List of Python dictionaries or objects to write.
        append (bool): If True, append to existing file. Otherwise, overwrite.
    """
    mode = 'a' if append else 'w'
    with open(filepath, mode, encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [5]:
def get_format_to_gpt(ci_output: ChatInterfaceOutputFormat):
    system_output = ci_output['system_response']
    state = ci_output['state']
    current_retrieval_results = ci_output['current_retrieval_results']

    return f"""SYSTEM OUTPUT:
```{system_output}```

STATE:
```{state}```

RETRIEVED DATA BY THE SYSTEM:
```{convert_multi_retriever_results_to_str(current_retrieval_results)}```
"""

In [6]:
def get_initial_prompt_to_chatgpt(domain: str, question: str):
    domain_expert_desc = f"a {domain} domain expert"
    if domain == "archeology":
        domain_expert_desc = "a domain expert in world cities, roman cities, radiocarbon data, world conflicts, and climate measurement exploration"
    return f"""You are simulating {domain_expert_desc}, who is interacting with a data assistant system to explore insights from an enterprise DATA_SOURCES. The system represents your information need as a set of target schemas, representing relevant table(s) for your question, along with a list of SQL statements, which if run sequentially on the (materialized) target schemas, will result in the answer to your question. You can criticize this representation if you think it does not represent your need correctly.

In this scenario, the system already has access to internal environment-related DATA_SOURCESs. You (the simulated user) are already somewhat familiar with the topics of the DATA_SOURCES, as it is commonly used in your team or organization. You are not uploading a new DATA_SOURCES or asking about the existence of some DATA_SOURCES. Your task is to gradually explore or refine your information need about some aspect of the data. You do not begin with a precise question; rather, your curiosity evolves based on system responses and your domain expertise.

Here is a possible eventual goal (you do not know this yet, but may arrive at it through exploration):

{question}

Your behavior should reflect the following:
- You are familiar with the domain.
- You explore and refine your question step-by-step depending on the system's ability to surface relevant information.
- You may be vague or even explore tangents, just as a curious analyst would when exploring data without a clear goal.
- You will only arrive at the specific question above if the system's output correctly leads you there.

Continue your role as the domain expert. This is the conversation so far (again, provide response as if you are prompting the system directly):

YOU: {INITIAL_PROMPT}"""

# INTERACTION

In [7]:
from processor.model.llm_message import LLMMessage, Role
from processor.model.option import LLMOption

In [8]:
index = 11
iteration = -1
INITIAL_PROMPT = benchmark[index]["interactive_initial_prompt"]
ITERATION_LIMIT = 15
print(f"Original (direct) question: {benchmark[index]["original_direct_question"]}")

Original (direct) question: Count the number of human conflicts between 800 and 1400 AD, and attribute them as best you can to modern countries. Define a conflict as between two actors that lasts at least a year.


In [9]:
gpt_init_prompt = get_initial_prompt_to_chatgpt(
    DATA_SOURCES[0], benchmark[index]["original_direct_question"]
)
gpt_messages = [LLMMessage(role=Role.SYSTEM.value, content=gpt_init_prompt)]
curr_user_prompt = INITIAL_PROMPT
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")

=> CURRENT USER PROMPT: I’m starting to look into historical conflicts across different regions, particularly during the medieval and early modern periods. It’d be great to have a system that could help me pull together events where human groups were in conflict, so I can explore patterns and impacts. I’d also like to attribute these conflicts to modern-day countries where possible. Can you walk me through what data is available and help me identify some starting points?


In [10]:
iteration += 1
system_output = chat_interface.process_user_input(curr_user_prompt)
print(f"===> SYSTEM OUTPUT: {system_output}")
format_to_gpt = get_format_to_gpt(system_output)
if iteration == 0:
    gpt_messages[0]['content'] += f"\n{format_to_gpt}"
else:
    gpt_messages.append(LLMMessage(role=Role.USER.value, content=format_to_gpt))
updated_user_prompt = gpt.chat(gpt_messages, LLMOption(temperature=0))
# updated_user_prompt = "YOU: Yes, let's proceed with materializing the Roman cities dataset. I want to see the sources listed in the 'select_bibliography' column and understand how many unique sources we have. Please run the necessary queries to extract this information."
gpt_messages.append(LLMMessage(role=Role.ASSISTANT.value, content=updated_user_prompt))
if updated_user_prompt.startswith("YOU:"):
    updated_user_prompt = updated_user_prompt[4:]
    updated_user_prompt = updated_user_prompt.strip()
curr_user_prompt = updated_user_prompt
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")

[2025-08-02 17:00:09] INFO in llm_conductor: Processing human input: I’m starting to look into historical conflicts across different regions, particularly during the medieval and early modern periods. It’d be great to have a system that could help me pull together events where human groups were in conflict, so I can explore patterns and impacts. I’d also like to attribute these conflicts to modern-day countries where possible. Can you walk me through what data is available and help me identify some starting points?


Loading checkpoint shards: 100%|██████████| 5/5 [00:04<00:00,  1.07it/s]


QWEN: response: {
    "intent": "tool_call",
    "tool": "ir_system",
    "args": {
        "prompt": "Tables related to historical conflicts, medieval and early modern periods, and modern-day country attributions"
    }
}
[2025-08-02 17:00:20] INFO in llm_conductor: IR System request with params: {'prompt': 'Tables related to historical conflicts, medieval and early modern periods, and modern-day country attributions'}
[2025-08-02 17:00:20] INFO in lm_interface: Starting document retrieval for prompt: Tables related to historical conflicts, medieval and early modern periods, and modern-day country at...
[2025-08-02 17:00:20] INFO in lm_interface: Selected retrievers: [<RetrieverType.PNEUMA: 'Pneuma'>, <RetrieverType.KNOWLEDGE_BASE: 'Knowledge Base'>]
[2025-08-02 17:00:20] INFO in lm_interface: => Retrieving from: RetrieverType.PNEUMA


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Reducing increased_k from 50 to 18


/home/luthfi/miniconda3/envs/pneuma/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[2025-08-02 17:00:21] INFO in lm_interface: => Initial retrieval returned 5 documents:
[2025-08-02 17:00:21] INFO in lm_interface: ==> Table conflict_brecke:
col: conflict | startyear | endyear | fatalities | century | decade
sample row 1: Italy (Murder of Drogone of Apulia) | 1051 | 1051 | nan | 1000 | 1050
sample row 2: England (Insurrection of “Lords Appellants”) | 1387 | 1388 | nan | 1300 | 1380
sample row 3: Spain (Intervention of the Union in Aragon) | 1288 | 1288 | nan | 1200 | 1280
sample row 4: Russia-Polovcians | 1100 | 1100 | 200.0 | 1100 | 1100
sample row 5: Poland and Lithuania (Revolt of Duke of Oppeln) | 1396 | 1396 | nan | 1300 | 1390
[2025-08-02 17:00:21] INFO in lm_interface: ==> Table roman_cities:
col: primary_key | ancient_toponym | modern_toponym | province | country | barrington_atlas_rank | barrington_atlas_reference | start_date | end_date | longitude_x | latitude_y | select_bibliography
sample row 1: Hanson2016_651 | Narbo Martius | Narbonne | Gallia Narbonens

In [11]:
iteration += 1
system_output = chat_interface.process_user_input(curr_user_prompt)
print(f"===> SYSTEM OUTPUT: {system_output}")
format_to_gpt = get_format_to_gpt(system_output)
if iteration == 0:
    gpt_messages[0]['content'] += f"\n{format_to_gpt}"
else:
    gpt_messages.append(LLMMessage(role=Role.USER.value, content=format_to_gpt))
updated_user_prompt = gpt.chat(gpt_messages, LLMOption(temperature=0))
# updated_user_prompt = "YOU: Yes, let's proceed with materializing the Roman cities dataset. I want to see the sources listed in the 'select_bibliography' column and understand how many unique sources we have. Please run the necessary queries to extract this information."
gpt_messages.append(LLMMessage(role=Role.ASSISTANT.value, content=updated_user_prompt))
if updated_user_prompt.startswith("YOU:"):
    updated_user_prompt = updated_user_prompt[4:]
    updated_user_prompt = updated_user_prompt.strip()
curr_user_prompt = updated_user_prompt
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")

[2025-08-02 17:01:14] INFO in llm_conductor: Processing human input: Thank you for the overview of the available tables. I think the `conflict_brecke` table is a good starting point since it directly relates to historical conflicts. I’m particularly interested in conflicts that occurred between 800 and 1400 AD. Can you help me filter the data from this table to focus on that specific time range? Additionally, I’d like to see how many conflicts there are in total within that period. 

Let's start with that. (Note: please check the current state (target schemas & sqls), if already defined, are they still relevant, or do they need any adjustments? For sqls, ensure all queries use ONLY available columns in the target schemas, so we do not run into errors.)
QWEN: response: {
    "intent": "internal_reasoning",
    "message": "The user has specified an interest in the `conflict_brecke` table and wants to filter conflicts between 800 and 1400 AD. I need to define a target schema for this tabl

In [ ]:
iteration += 1
system_output = chat_interface.process_user_input(curr_user_prompt)
print(f"===> SYSTEM OUTPUT: {system_output}")
format_to_gpt = get_format_to_gpt(system_output)
if iteration == 0:
    gpt_messages[0]['content'] += f"\n{format_to_gpt}"
else:
    gpt_messages.append(LLMMessage(role=Role.USER.value, content=format_to_gpt))
updated_user_prompt = gpt.chat(gpt_messages, LLMOption(temperature=0))
# updated_user_prompt = "YOU: Yes, let's proceed with materializing the Roman cities dataset. I want to see the sources listed in the 'select_bibliography' column and understand how many unique sources we have. Please run the necessary queries to extract this information."
gpt_messages.append(LLMMessage(role=Role.ASSISTANT.value, content=updated_user_prompt))
if updated_user_prompt.startswith("YOU:"):
    updated_user_prompt = updated_user_prompt[4:]
    updated_user_prompt = updated_user_prompt.strip()
curr_user_prompt = updated_user_prompt
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")

[2025-08-02 17:02:09] INFO in llm_conductor: Processing human input: The queries look good! Please go ahead and execute them to get the count of conflicts between 800 and 1400 AD, as well as the details of those conflicts. I'm particularly interested in understanding the scope and nature of these conflicts. (Note: please check the current state (target schemas & sqls), if already defined, are they still relevant, or do they need any adjustments? For sqls, ensure all queries use ONLY available columns in the target schemas, so we do not run into errors.)
QWEN: response: {
    "intent": "tool_call",
    "tool": "sql_engine",
    "args": ""
}
[2025-08-02 17:02:11] INFO in llm_conductor: SQL Engine called


# FINAL

In [ ]:
write_jsonl(f"benchmark_data/{DATA_SOURCES}_{index+1}.jsonl", gpt_messages, True)